# 02 — Feature Engineering
## Global Job Market Compensation Analysis

**Objective:** Create business-meaningful features that genuinely improve analysis and modeling -- not a checklist of "common" features padded on for appearance.

**Critical rule:** several candidate features are derived directly from `salary` (bands, ratios, indicators). These are useful for EDA and as classification targets, but using them as **predictors when modeling salary itself is target leakage** and is explicitly avoided in Phase 7. Each such feature is flagged clearly below.

**Explicitly skipped, with reasoning:**
- `Demand Score` -- no real labor-demand signal exists in this dataset (no postings/applicants/time-to-fill data). Fabricating one from row counts would misrepresent sample size as a business metric.
- `Salary Growth Category` -- this data is cross-sectional (one row per person-instance), not a panel tracking the same individual over time, so individual salary growth cannot be computed. Aggregate year-over-year trend belongs in EDA (Phase 5), not here.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', None)

df = pd.read_parquet(r"C:\Users\rashm\Downloads\Job Market\data\processed\job_market_cleaned.parquet")
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded cleaned dataset: 499,972 rows x 13 columns


,country,city,occupation,field,years_of_experience,salary,employment_type,education_level,gender,company_size,year,month,salary_outlier_flag
0,Switzerland,Zurich,Operations Manager,Operations,16,359609,work_from_home,Master,Male,Large,2023,8,False
1,India,Bangalore,HR Analyst,Human Resources,12,79059,part_time,PhD,Male,Large,2023,5,False
2,Sweden,Stockholm,Software Engineer,Technology,10,258077,freelance,Master,Male,Enterprise,2023,9,False
3,South Korea,Seoul,Operations Manager,Operations,7,252282,part_time,Master,Female,Large,2024,12,False
4,United States,New York,Cloud Engineer,Technology,4,330618,part_time,Master,Male,Enterprise,2022,1,False


## 1. Experience Level (safe predictor)

Buckets `years_of_experience` (0-25) into four interpretable career stages. This is a standard, defensible business segmentation -- not derived from salary, so it's a clean regression/classification predictor.

In [2]:
def experience_level(years):
    if years <= 3:
        return 'Junior'
    elif years <= 8:
        return 'Mid'
    elif years <= 15:
        return 'Senior'
    else:
        return 'Executive'

df['experience_level'] = df['years_of_experience'].apply(experience_level)
df['experience_level'] = pd.Categorical(
    df['experience_level'], categories=['Junior', 'Mid', 'Senior', 'Executive'], ordered=True
)

df['experience_level'].value_counts().sort_index()


experience_level
Junior        81042
Mid          101485
Senior       141605
Executive    175840
Name: count, dtype: int64

## 2. Quarter (safe predictor)

Coarser time grouping than `month` -- reduces a 12-level categorical to 4, useful for trend visuals and avoids sparsity in group-level breakdowns later.

In [3]:
df['quarter'] = pd.cut(
    df['month'], bins=[0,3,6,9,12], labels=['Q1','Q2','Q3','Q4']
)
df['quarter'].value_counts().sort_index()


quarter
Q1    124678
Q2    125035
Q3    125085
Q4    125174
Name: count, dtype: int64

## 3. Region (safe predictor)

Groups the 21 countries into continents/regions. This lets us test regional effects (e.g., "Europe vs. APAC") without a 21-level dummy explosion in linear models, and gives Power BI a clean top-level filter.

In [4]:
region_map = {
    'United States': 'North America', 'Canada': 'North America', 'Mexico': 'North America',
    'Brazil': 'South America',
    'United Kingdom': 'Europe', 'Germany': 'Europe', 'France': 'Europe', 'Spain': 'Europe',
    'Italy': 'Europe', 'Netherlands': 'Europe', 'Sweden': 'Europe', 'Ireland': 'Europe',
    'Switzerland': 'Europe',
    'India': 'Asia', 'Singapore': 'Asia', 'Japan': 'Asia', 'South Korea': 'Asia',
    'United Arab Emirates': 'Middle East',
    'South Africa': 'Africa',
    'Australia': 'Oceania', 'New Zealand': 'Oceania',
}

df['region'] = df['country'].map(region_map).astype('category')

# validation: every country must map to a region
unmapped = df[df['region'].isna()]['country'].unique()
assert len(unmapped) == 0, f"Unmapped countries found: {unmapped}"

df['region'].value_counts()


region
Europe           205283
North America     90965
Asia              90385
Oceania           45100
South America     23010
Middle East       22624
Africa            22605
Name: count, dtype: int64

## 4. Salary Band -- EDA / classification target ONLY (leakage-flagged)

Quartile-based Low / Mid / High / Very High banding of `salary`. **This feature must never be used as a predictor when modeling salary itself** -- it's derived directly from the target. Its legitimate uses are: (a) EDA cross-tabs, (b) the target variable for the Phase 7 classification task ("predict salary category").

In [5]:
df['salary_band'] = pd.qcut(
    df['salary'], q=4, labels=['Low', 'Mid', 'High', 'Very High']
)

print("salary_band distribution:")
print(df['salary_band'].value_counts().sort_index())
print()
print("LEAKAGE WARNING: salary_band is derived from salary.")
print("Use only for EDA and as the classification target -- never as a regression predictor.")


salary_band distribution:
salary_band
Low          124994
Mid          124995
High         124993
Very High    124990
Name: count, dtype: int64

LEAKAGE WARNING: salary_band is derived from salary.
Use only for EDA and as the classification target -- never as a regression predictor.


## 5. High Salary Indicator -- EDA / alternate classification target ONLY (leakage-flagged)

Binary flag: is this salary above the median for its *own country*? (Country-relative, not global -- a flat global median would just reproduce the country ranking we already saw in Phase 2, telling us nothing new.) Same leakage warning as above applies.

In [6]:
country_median = df.groupby('country', observed=True)['salary'].transform('median')
df['high_salary_indicator'] = (df['salary'] > country_median).astype(int)

print(df['high_salary_indicator'].value_counts(normalize=True))
print()
print("LEAKAGE WARNING: high_salary_indicator is derived from salary.")
print("Use only for EDA and as an alternate classification target -- never as a regression predictor.")


high_salary_indicator
0    0.500016
1    0.499984
Name: proportion, dtype: float64

LEAKAGE WARNING: high_salary_indicator is derived from salary.
Use only for EDA and as an alternate classification target -- never as a regression predictor.


## 6. Salary per Experience Year -- EDA ONLY (leakage-flagged, descriptive)

A "pay efficiency" ratio: salary earned per year of experience. Useful for spotting roles/countries where experience is compensated efficiently vs. not -- but it's arithmetically derived from `salary`, so it is descriptive only and never a model input for salary prediction. We add 1 to the denominator to avoid division by zero for 0-experience rows.

In [7]:
df['salary_per_experience_year'] = df['salary'] / (df['years_of_experience'] + 1)

print(df['salary_per_experience_year'].describe())
print()
print("LEAKAGE WARNING: derived from salary. EDA-only, never a regression predictor.")


count    499972.000000
mean      24971.149085
std       25382.377148
min        1578.269231
25%       11293.228632
50%       16818.181818
75%       27977.491071
max      370000.000000
Name: salary_per_experience_year, dtype: float64

LEAKAGE WARNING: derived from salary. EDA-only, never a regression predictor.


## 7. Top Occupation Indicator (safe predictor, with a train/test caveat)

Flags whether an occupation's *mean salary* falls in the top quartile of occupation-level mean salaries. This is aggregated at the occupation level (12 occupations), not leaking individual salary into its own row -- but it is still computed from the salary column in aggregate, so when we build train/test splits in Phase 7, this must be recomputed from the **training fold only** and then mapped onto the test fold, or it silently leaks test-set information back into training. Flagging that now so it isn't forgotten three phases from now.

In [8]:
occupation_mean_salary = df.groupby('occupation', observed=True)['salary'].mean().sort_values(ascending=False)
print("Occupation mean salaries (descending):")
print(occupation_mean_salary)

top_quartile_cutoff = occupation_mean_salary.quantile(0.75)
top_occupations = occupation_mean_salary[occupation_mean_salary >= top_quartile_cutoff].index.tolist()

df['top_occupation_indicator'] = df['occupation'].isin(top_occupations).astype(int)

print()
print("Top-quartile occupations:", top_occupations)
print()
print(df['top_occupation_indicator'].value_counts(normalize=True))
print()
print("CAVEAT: recompute this from the training fold only in Phase 7 to avoid train/test leakage.")


Occupation mean salaries (descending):
occupation
AI Engineer             252146.871926
Product Manager         251254.229331
Cloud Engineer          242811.519409
Data Scientist          233242.205098
Software Engineer       223667.326878
Operations Manager      213447.475852
Financial Analyst       202581.415425
UX Designer             192173.979590
Business Analyst        191619.731466
Data Analyst            179691.180656
Marketing Specialist    156345.184921
HR Analyst              144604.000337
Name: salary, dtype: float64

Top-quartile occupations: ['AI Engineer', 'Product Manager', 'Cloud Engineer']

top_occupation_indicator
0    0.749336
1    0.250664
Name: proportion, dtype: float64

CAVEAT: recompute this from the training fold only in Phase 7 to avoid train/test leakage.


## 8. Save Feature-Engineered Dataset

In [9]:
OUT_PATH = r"C:\Users\rashm\Downloads\Job Market\data\processed\job_market_features.parquet"
df.to_parquet(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
print("New columns added:")
new_cols = ['experience_level', 'quarter', 'region', 'salary_band',
            'high_salary_indicator', 'salary_per_experience_year', 'top_occupation_indicator']
print(new_cols)
df[new_cols].head()


Saved: C:\Users\rashm\Downloads\Job Market\data\processed\job_market_features.parquet
Shape: 499,972 rows x 20 columns

New columns added:
['experience_level', 'quarter', 'region', 'salary_band', 'high_salary_indicator', 'salary_per_experience_year', 'top_occupation_indicator']


,experience_level,quarter,region,salary_band,high_salary_indicator,salary_per_experience_year,top_occupation_indicator
0,Executive,Q3,Europe,Very High,1,21153.470588,0
1,Senior,Q2,Asia,Low,0,6081.461538,0
2,Senior,Q3,Europe,High,1,23461.545455,0
3,Mid,Q4,Asia,High,1,31535.250000,0
4,Mid,Q1,North America,Very High,1,66123.600000,1
